## Extraer artefactos del modelo a implementar en Producción

Este notebook extrae 4 componentes del modelo final:<br>
1. item.map: Mapeo de Item_Index con el Item_ID original.
2. user.map: Mpaeo de User_Index con el User_ID original.
3. item_factors: Los 24 factores generados para cada item dentro del modelo entrenado.
3. user_factors: Los 24 factores generados para cada user dentro del modelo entrenado.

### Item & User maps

In [3]:
import pandas as pd

item_map = pd.read_parquet("datasets/parquet/steam/reviews/processed/item_map.parquet")
user_map = pd.read_parquet("datasets/parquet/steam/reviews/processed/user_map.parquet")

print(item_map.columns.tolist(), item_map.shape)
print(item_map.head())
print(user_map.columns.tolist(), user_map.shape)
print(user_map.head())

['item_id', 'item_idx'] (78376, 2)
   item_id  item_idx
0       10         0
1       20         1
2       30         2
3       40         3
4       50         4
['user_id', 'user_idx'] (17049073, 2)
             user_id  user_idx
0  76561197960265745         0
1  76561197960265763         1
2  76561197960265778         2
3  76561197960265781         3
4  76561197960265822         4


In [4]:
import numpy as np

data = np.load("models/Tunned/als_model_prod.npz", allow_pickle=True)
print(data.files)  # confirmar nombres exactos dentro del npz

item_factors = data["item_factors"].astype(np.float32)
user_factors = data["user_factors"].astype(np.float32)

assert len(item_map) == item_factors.shape[0], \
    f"Mismatch: item_map tiene {len(item_map)} filas, item_factors tiene {item_factors.shape[0]}"
assert len(user_map) == user_factors.shape[0], \
    f"Mismatch: user_map tiene {len(user_map)} filas, user_factors tiene {user_factors.shape[0]}"

assert sorted(item_map["item_idx"].tolist()) == list(range(len(item_map))), \
    "item_idx no es un rango contiguo 0..n-1"
assert sorted(user_map["user_idx"].tolist()) == list(range(len(user_map))), \
    "user_idx no es un rango contiguo 0..n-1"

print("Shapes y rangos OK")

['user_factors', 'item_factors', 'regularization', 'factors', 'num_threads', 'iterations', 'use_native', 'use_cg', 'cg_steps', 'calculate_training_loss', 'dtype', 'random_state', 'alpha']
Shapes y rangos OK


In [5]:
item_map_sorted = item_map.sort_values("item_idx").reset_index(drop=True)
user_map_sorted = user_map.sort_values("user_idx").reset_index(drop=True)

assert (item_map_sorted["item_idx"].values == np.arange(len(item_map_sorted))).all()
assert (user_map_sorted["user_idx"].values == np.arange(len(user_map_sorted))).all()

item_ids = item_map_sorted["item_id"].values   # array: posición = idx del modelo, valor = app_id real
user_ids = user_map_sorted["user_id"].values   # array: posición = idx del modelo, valor = SteamID64 real

In [6]:
import os

os.makedirs("artifacts", exist_ok=True)

item_map.to_parquet("artifacts/item_map.parquet", index=False)
user_map.to_parquet("artifacts/user_map.parquet", index=False)

print(os.path.getsize("artifacts/item_map.parquet") / 1024, "KB")
print(os.path.getsize("artifacts/user_map.parquet") / 1024 / 1024, "MB")

949.318359375 KB
147.91630458831787 MB


### Vectores de 24 dimensiones (Factores)

In [7]:
import pandas as pd
import numpy as np

games_df = pd.read_parquet("datasets/parquet/steam/games/games.parquet")

# Ajustar "AppID" al nombre real de columna en este parquet
valid_appids = set(games_df["AppID"].tolist())

# item_ids ya está cargado y alineado posicionalmente con item_factors
is_valid_game = np.array([appid in valid_appids for appid in item_ids], dtype=bool)

print(f"Válidos: {is_valid_game.sum()} / {len(is_valid_game)}")

Válidos: 58638 / 78376


In [8]:
np.savez_compressed(
    "artifacts/item_factors.npz",
    item_factors=item_factors,
    item_ids=item_ids,
    is_valid_game=is_valid_game, # Se agrega flag para separar juegos de DLC/Herramientas
)

import os
print(f"Tamaño: {os.path.getsize('artifacts/item_factors.npz') / 1024 / 1024:.2f} MB")

Tamaño: 6.83 MB


In [9]:
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import os

# user_factors: (17,049,073 x 24)
# user_ids: array de SteamID64

user_factors_as_lists = [row.tolist() for row in user_factors]

n_users, n_factors = user_factors.shape

flat_values = pa.array(user_factors.reshape(-1), type=pa.float32())

# Offsets: cada fila empieza en múltiplos de 24 (todas las filas tienen igual longitud)
offsets = pa.array(np.arange(0, (n_users + 1) * n_factors, n_factors), type=pa.int32())

# ListArray genera un ARRAY<FLOAT32> plano
factors_list_array = pa.ListArray.from_arrays(offsets, flat_values)

id_array = pa.array(user_ids.astype(str))

table = pa.table({
    "user_id": id_array,
    "user_factors": factors_list_array,
})

pq.write_table(table, "artifacts/user_factors.parquet", compression="snappy")

print(f"Tamaño: {os.path.getsize('artifacts/user_factors.parquet') / 1024 / 1024:.0f} MB")

Tamaño: 1663 MB
